In [10]:
%pip install groq pyarrow requests huggingface_hub

  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 7.2 MB/s eta 0:00:00 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 19.3 MB/s eta 0:00:00m eta 0:00:010:01:01
Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (807 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 kB 10.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [14]:
import json
import numpy as np
import requests 
import pandas as pd 
from collections import Counter
import ast 
from groq import Groq

In [17]:
#----------------
# This is the LLM stuff
#----------------

def query_ollama(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": "llama3", "prompt": prompt, "stream": False}
    )
    return response.json()

def query_groq(prompt):

    client = Groq(api_key="") # Shh don't tell anybody
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

def parse_response(response):
    if isinstance(response, list):
        return [r for r in response if isinstance(r, str)]
    if isinstance(response, str):
        try:
            results = ast.literal_eval(response)
        except (ValueError, SyntaxError):
            return []
        translated = []
        for emotion in results:
            if isinstance(emotion, str) and emotion in inverted_labels:
                translated.append(inverted_labels[emotion])
            # silently skip unknown emotions like 'desperation', 'compassion', 'hope'
        return translated
    return []


def make_llm_results_dfs(results):
    label_cols = list(range(28))
    pred_cols  = list(range(100, 128))
    cols       = label_cols + pred_cols

    ollama_df = pd.DataFrame(0, index=list(results.keys()), columns=cols)
    groq_df   = pd.DataFrame(0, index=list(results.keys()), columns=cols)

    for ident, row in results.items():
        ollama_preds = parse_response(row.get('ollama', '[]'))
        groq_preds   = parse_response(row.get('groq',   '[]'))

        for pred in ollama_preds:
            ollama_df.at[ident, 100 + pred] = 1

        for pred in groq_preds:
            groq_df.at[ident, 100 + pred] = 1

        for label in row['labels']:
            if label in label_cols:
                ollama_df.at[ident, label] = 1
                groq_df.at[ident, label]   = 1

    return ollama_df, groq_df

def get_report(df, model_name, label_mappings):
    y_true = df[list(range(28))].values
    y_pred = df[list(range(100, 128))].values

    # Drop all-zero rows
    mask   = (y_true.sum(axis=1) > 0) | (y_pred.sum(axis=1) > 0)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    target_names = [label_mappings[i] for i in range(28)]

    report = classification_report(
        y_true, y_pred,
        target_names=target_names,
        zero_division=0,
        output_dict=True
    )

    macro_f1    = f1_score(y_true, y_pred, average='macro',  zero_division=0)
    micro_f1    = f1_score(y_true, y_pred, average='micro',  zero_division=0)
    exact_match = (y_true == y_pred).all(axis=1).mean()

    print(f"\n--- {model_name} ---")
    print(f"Exact match: {exact_match:.4f}")
    print(f"Macro F1:    {macro_f1:.4f}")
    print(f"Micro F1:    {micro_f1:.4f}")
    print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))

    return report, {"exact_match": exact_match, "macro_f1": macro_f1, "micro_f1": micro_f1}

# I do so love learning things the hard way. You'd think at my age I'd have gotten 'save your work.' You would, however, be wrong.
# Claude wrote this btw

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

def save_work(filename, stuff_to_save):
    with open(filename, 'w') as f:
        json.dump(stuff_to_save, f, cls=NumpyEncoder)
    

In [11]:
#----------------
# Dataset
#----------------
splits = {'train': 'simplified/train-00000-of-00001.parquet', 'validation': 'simplified/validation-00000-of-00001.parquet', 'test': 'simplified/test-00000-of-00001.parquet'}
train_df = pd.read_parquet("hf://datasets/google-research-datasets/go_emotions/" + splits["train"])
test_df = pd.read_parquet("hf://datasets/google-research-datasets/go_emotions/" + splits["test"])
# df_llm = df_llm.iloc[:400, :] # This was for the Groq run, which crapped out even at this level

all_labels = test_df["labels"].explode()
NUM_LABELS = int(all_labels.max()) + 1 # I really don't remember why I added 1 but there was some problem and this fixed it
label_mappings = {0: 'admiration',1: 'amusement',2: 'anger',3: 'annoyance',4: 'approval',5: 'caring',6: 'confusion',7: 'curiosity',
                  8: 'desire',9: 'disappointment',10: 'disapproval',11: 'disgust',12: 'embarrassment',13: 'excitement',14: 'fear',
                  15: 'gratitude',16: 'grief',17: 'joy',18: 'love',19: 'nervousness',20: 'optimism',21: 'pride',22: 'realization',
                  23: 'relief',24: 'remorse',25: 'sadness',26: 'surprise',27: 'neutral'}
emo_list = [v for k,v in label_mappings.items()]
inverted_labels = {v: k for k, v in label_mappings.items()}
pred_cols  = [f"Predicted_{label_mappings[i]}"  for i in range(NUM_LABELS)]
label_cols = [f"Truth_{label_mappings[i]}" for i in range(NUM_LABELS)]

#----------------
# Dataset
#----------------
splits = {'train': 'simplified/train-00000-of-00001.parquet', 'validation': 'simplified/validation-00000-of-00001.parquet', 'test': 'simplified/test-00000-of-00001.parquet'}
train_df = pd.read_parquet("hf://datasets/google-research-datasets/go_emotions/" + splits["train"])
test_df = pd.read_parquet("hf://datasets/google-research-datasets/go_emotions/" + splits["test"])
# df_llm = df_llm.iloc[:400, :] # This was for the Groq run, which crapped out even at this level

/home/jj/PycharmProjects/datamining/datamining/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#----------------
# Querying
#----------------
# This runs the  dataset against both LLMs 
# Since this is one shot it doesn't matter what dataset to use. I chose test because it's shorter

results = {}
for _, row in test_df.iterrows():
    text = row['text']
    labels = row['labels']
    ident = row['id']
    prompt = f"""for the text on the next line, clasify it by giving a list of items in this list{','.join(emo_list)}. Send back only the items in this list that apply to the text in a python-style list with no other text returned
        {text}"""
    results[ident] = {}
    results[ident]['labels'] = labels
    results[ident]['ollama'] = query_ollama(prompt)['response']
    results[ident]['groq'] = [0,0] # query_groq(prompt) # had to run them separately becuase of the token issue. This was annoying
print("done")

save_work('ollama_results.json', results)
# save_work('groq_results.json', results)

In [ ]:
#----------------
# Parses results and creates the reports
#----------------

ollama_df, groq_df = make_llm_results_dfs(ollama_results_load)
ollama_report, ollama_scores = get_report(ollama_df, "Ollama", label_mappings)

#groq_report, groq_scores = get_report(groq_df, "Groq", label_mappings)

In [ ]:
#----------------
# This is the BoW classifier
#----------------

# So I asked Claude what bag of words classifier I should use and it suggested this
# Interesting to implement. Wasn't familiar with Pipeline before this. Useful.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import ComplementNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer(classes=list(range(28)))

pipeline = Pipeline([
    ('tfidf',      TfidfVectorizer()),
    ('classifier', OneVsRestClassifier(ComplementNB())),
])
X_train = train_df['text'].values
X_test = test_df['text'].values
y_train = mlb.fit_transform(train_df['labels'])
y_test = mlb.fit_transform(test_df['labels'])
pipeline.fit(X_train, y_train) 
preds = pipeline.predict(X_test)

print(classification_report(
    y_test, preds,
    target_names=[label_mappings[i] for i in range(28)],
    zero_division=0
))

# ComplementNB
exact_match_nb = (y_test == preds).all(axis=1).mean()

print(f"ComplementNB exact match: {exact_match_nb:.4f}")

In [ ]:
#----------------
# Random
#----------------

label_probs = y_train.mean(axis=0)
random_preds = np.random.binomial(1, label_probs, size=y_test.shape)

print(classification_report(
    y_test, random_preds,
    target_names=[label_mappings[i] for i in range(28)],
    zero_division=0
))

exact_match_random = (y_test == random_preds).all(axis=1).mean()
print(f"Random exact match:       {exact_match_random:.4f}")

In [50]:
#----------------
# Random Majority
#----------------
from collections import defaultdict
unique_labels  = defaultdict(int)
# for label in train_df['labels']:
#     
per_label_counts = defaultdict(int)

for label in train_df['labels']:
    for individual_label in label:
        per_label_counts[individual_label] += 1
    label_str = str(sorted(label.tolist()))
    unique_labels[label_str] += 1
max_label = 0
for _, count in unique_labels.items(): 
    if count > max_label:
        max_label = count
right_percent = max_label/len(train_df)
print(right_percent)

for ilabel, count in per_label_counts.items():
    print(f'{label_mappings[ilabel]:<10}\t{count/len(train_df):.4f}')

0.2953927666436305
neutral   	0.3276
anger     	0.0361
fear      	0.0137
annoyance 	0.0569
surprise  	0.0244
gratitude 	0.0613
desire    	0.0148
optimism  	0.0364
admiration	0.0951
confusion 	0.0315
amusement 	0.0536
approval  	0.0677
caring    	0.0250
embarrassment	0.0070
realization	0.0256
disappointment	0.0292
grief     	0.0018
sadness   	0.0305
curiosity 	0.0505
joy       	0.0334
love      	0.0481
excitement	0.0196
disapproval	0.0466
remorse   	0.0126
disgust   	0.0183
relief    	0.0035
pride     	0.0026
nervousness	0.0038


In [18]:
print(majority_class)

0


In [19]:
print(label_counts)

Counter({0: 1, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 1, 11: 1, 12: 1, 13: 1, 14: 1, 15: 1, 16: 1, 17: 1, 18: 1, 19: 1, 20: 1, 21: 1, 22: 1, 23: 1, 24: 1, 25: 1, 26: 1, 27: 1})


In [ ]:
label_dist = pd.DataFrame()
label_cols = list(range(28))
for label in row['labels']:
    if label in label_cols:
        label_dist.at[ident, label] = 1
